# Training the Nano DiT

In this notebook we train our miniature Diffusion Transformer (NanoDiT) on a synthetic moving-shapes dataset. By the end, you will have a trained checkpoint that can generate short videos from noise.

The training procedure follows the same pattern used by Wan 2.1 and other modern video diffusion models:

1. Sample a batch of training videos
2. Encode them to latent space with the VAE
3. Sample a random timestep $t \sim \mathcal{U}(0, 1)$
4. Add noise: $x_t = (1 - \sigma) \cdot x_0 + \sigma \cdot \epsilon$
5. Predict the velocity: $\hat{v} = \text{DiT}(x_t, t, \text{context})$
6. Minimize MSE: $\mathcal{L} = \|\hat{v} - (\epsilon - x_0)\|^2$

## Prerequisites

You need a video dataset to train on. There are two options:

**Option A: Generate synthetic data** (recommended for this tutorial)

```bash
# From the project root directory:
python -m nano_video_gen.data.generate_synthetic
```

This creates 60 short videos of geometric shapes (circles, squares) moving in various directions, with corresponding text prompts.

**Option B: Download the DiffSynth example dataset**

```bash
# If you have modelscope installed:
modelscope download --dataset DiffSynth-Studio/examples_video --local_dir ./data/examples_video
```

The code below will automatically generate synthetic data if none is found.

In [ ]:
import sys
import os

# Add the project root to Python path so we can import nano_video_gen
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from nano_video_gen.model.nano_dit import NanoDiT
from nano_video_gen.model.nano_vae import DummyVAE
from nano_video_gen.diffusion.flow_match import FlowMatchScheduler
from nano_video_gen.data.dataset import VideoDataset, SimpleTextEncoder
from nano_video_gen.visualization.viz import plot_training_curves, save_video_grid

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Prepare the Dataset

First, we generate the synthetic dataset if it does not already exist, then wrap it in a `DataLoader` for batched training.

In [ ]:
data_dir = os.path.join(os.getcwd(), '..', 'data', 'synthetic_dataset')

if not os.path.exists(data_dir):
    print("Synthetic dataset not found. Generating...")
    from nano_video_gen.data.generate_synthetic import generate_dataset
    generate_dataset(output_dir=data_dir)
else:
    print(f"Dataset already exists at: {data_dir}")

In [ ]:
# Create the dataset and dataloader
dataset = VideoDataset(
    base_path=data_dir,
    height=64,
    width=64,
    num_frames=16,
)

dataloader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    num_workers=0,  # Keep 0 for notebook compatibility
    drop_last=True,
)

print(f"Dataset size: {len(dataset)} videos")
print(f"Number of unique prompts: {dataset.num_prompts}")
print(f"Batches per epoch: {len(dataloader)}")
print()

# Inspect a sample batch
sample_batch = next(iter(dataloader))
print(f"Batch video shape:      {sample_batch['video'].shape}")    # [B, 3, 16, 64, 64]
print(f"Batch prompt_idx shape: {sample_batch['prompt_idx'].shape}")  # [B]
print(f"Sample prompts: {sample_batch['prompt'][:2]}")

## 2. Initialize the Models

We need three components:

| Component | Purpose | Trained? |
|-----------|---------|----------|
| **NanoDiT** | Predicts the velocity (denoising direction) | Yes |
| **DummyVAE** | Compresses video to/from latent space | Yes (weights update via encode path) |
| **SimpleTextEncoder** | Maps prompt indices to text embeddings | Yes |

In a production setup (e.g., Wan 2.1), the VAE and text encoder are typically pretrained and frozen. Here, since our DummyVAE has no pretrained weights, its encoder runs in `torch.no_grad()` mode inside `compute_loss` (so it does not receive gradient updates), while the decoder is not used during training at all. The `SimpleTextEncoder` is trained jointly with the DiT.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize NanoDiT
model = NanoDiT(
    dim=128,
    in_dim=4,
    ffn_dim=512,
    out_dim=4,
    text_dim=64,
    freq_dim=64,
    num_heads=4,
    num_layers=2,
    patch_size=(1, 2, 2),
).to(device)

# Initialize DummyVAE
vae = DummyVAE(
    in_channels=3,
    latent_channels=4,
    spatial_factor=4,
    temporal_factor=4,
).to(device)

# Initialize SimpleTextEncoder
text_encoder = SimpleTextEncoder(
    num_prompts=dataset.num_prompts,
    text_dim=64,
    seq_len=8,
).to(device)

# Print parameter counts
def count_params(m):
    return sum(p.numel() for p in m.parameters())

print(f"\nParameter counts:")
print(f"  NanoDiT:          {count_params(model):>10,}")
print(f"  DummyVAE:         {count_params(vae):>10,}")
print(f"  SimpleTextEncoder: {count_params(text_encoder):>10,}")
print(f"  Total:            {count_params(model) + count_params(vae) + count_params(text_encoder):>10,}")

## 3. Training Loop

The training loop follows the flow matching paradigm. At each step:

1. **Encode**: Pass the video through the VAE encoder to get latent $x_0$
2. **Sample timestep**: Draw $t \sim \mathcal{U}(0, 1)$ and compute $\sigma = t$
3. **Add noise**: Compute $x_t = (1 - \sigma) \cdot x_0 + \sigma \cdot \epsilon$ where $\epsilon \sim \mathcal{N}(0, I)$
4. **Predict velocity**: $\hat{v} = \text{DiT}(x_t, t \cdot 1000, \text{context})$
5. **Compute loss**: $\mathcal{L} = \text{MSE}(\hat{v},\ \epsilon - x_0)$

All of these steps are encapsulated in `scheduler.compute_loss(...)` for convenience.

We use **AdamW** with weight decay for stable training, and **gradient clipping** to prevent exploding gradients.

In [ ]:
# Training configuration
num_epochs = 50
learning_rate = 1e-3
weight_decay = 0.01
max_grad_norm = 1.0

# Set up scheduler, optimizer
scheduler = FlowMatchScheduler()
scheduler.set_timesteps(50, shift=5.0)

# Optimize both the DiT model and the text encoder jointly
trainable_params = list(model.parameters()) + list(text_encoder.parameters())
optimizer = torch.optim.AdamW(trainable_params, lr=learning_rate, weight_decay=weight_decay)

# Training loop
model.train()
text_encoder.train()
vae.eval()  # VAE is used for encoding only (no grad)

losses = []
epoch_losses = []

# Create output directory for checkpoints
output_dir = os.path.join(os.getcwd(), '..', 'outputs')
os.makedirs(output_dir, exist_ok=True)

for epoch in range(num_epochs):
    epoch_loss_sum = 0.0
    epoch_steps = 0

    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False)

    for batch in progress_bar:
        video = batch["video"].to(device)
        prompt_idx = batch["prompt_idx"].to(device)

        # Get text embeddings from the simple encoder
        context = text_encoder(prompt_idx)

        # Compute flow matching loss
        # (internally: VAE encode -> add noise -> predict velocity -> MSE)
        loss = scheduler.compute_loss(model, video, context, vae=vae)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable_params, max_grad_norm)
        optimizer.step()

        # Track loss
        loss_val = loss.item()
        losses.append(loss_val)
        epoch_loss_sum += loss_val
        epoch_steps += 1

        progress_bar.set_postfix(loss=f"{loss_val:.4f}")

    avg_epoch_loss = epoch_loss_sum / max(epoch_steps, 1)
    epoch_losses.append(avg_epoch_loss)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/{num_epochs}: avg_loss={avg_epoch_loss:.4f}, last_loss={losses[-1]:.4f}")

        # Save intermediate checkpoint
        ckpt_path = os.path.join(output_dir, f"checkpoint_epoch{epoch+1}.pt")
        torch.save({
            "model": model.state_dict(),
            "vae": vae.state_dict(),
            "text_encoder": text_encoder.state_dict(),
            "epoch": epoch + 1,
            "losses": losses,
        }, ckpt_path)
        print(f"  -> Saved checkpoint: {ckpt_path}")

print("\nTraining complete!")

## 4. Training Loss Curves

Let's visualize how the loss evolved during training. We expect to see:
- A rapid decrease in the first few epochs as the model learns the basic structure
- Gradual convergence as the model refines its predictions

The `plot_training_curves` function shows both raw and exponentially-smoothed loss, plus a log-scale view.

In [ ]:
fig = plot_training_curves(losses, title="Flow Matching Training Loss")
plt.show()

## 5. What to Expect

With this tiny model (~1.5M parameters) on the synthetic moving-shapes dataset:

- **Loss should decrease noticeably** within the first 10--20 epochs. The synthetic data has strong, simple patterns (solid shapes on black backgrounds) that are relatively easy to learn.
- **On GPU**: Training 50 epochs should complete in under 10 minutes.
- **On CPU**: Expect ~30 minutes or more, depending on the hardware.
- **Generated outputs** will look rough but should show recognizable colored blobs moving in the trained directions. Do not expect photorealistic video from a 1.5M-parameter model!

The important takeaway is that the *architecture and training procedure are identical* to what Wan 2.1 uses at 14B parameters. Scale up the dimensions, use a real pretrained VAE, plug in a T5 text encoder, train on millions of videos, and you get state-of-the-art video generation.

## 6. Save Final Checkpoint

In [ ]:
final_ckpt_path = os.path.join(output_dir, "checkpoint_final.pt")

torch.save({
    "model": model.state_dict(),
    "vae": vae.state_dict(),
    "text_encoder": text_encoder.state_dict(),
    "epoch": num_epochs,
    "losses": losses,
    "num_prompts": dataset.num_prompts,
}, final_ckpt_path)

print(f"Final checkpoint saved to: {final_ckpt_path}")
print(f"  Model params:  {count_params(model):,}")
print(f"  Final loss:    {losses[-1]:.4f}")
print(f"  Total steps:   {len(losses)}")

## 7. Quick Inference Test

Let's verify that training worked by generating a single video from pure noise. We will:

1. Start with random noise in latent space: $x_T \sim \mathcal{N}(0, I)$
2. Iteratively denoise using the trained DiT
3. Decode the final latent back to pixel space with the VAE

This is a preview -- we will explore inference in much greater detail in the next notebook.

In [ ]:
model.eval()
text_encoder.eval()

with torch.no_grad():
    # Start from pure noise in latent space
    # Shape: [B, latent_channels, T_lat, H_lat, W_lat] = [1, 4, 4, 16, 16]
    x = torch.randn(1, 4, 4, 16, 16, device=device)

    # Get text conditioning for the first prompt
    ctx = text_encoder(torch.tensor([0], device=device))

    # Set up the denoising schedule
    scheduler.set_timesteps(20, shift=5.0)

    # Iterative denoising
    for t in tqdm(scheduler.timesteps, desc="Denoising"):
        v_pred = model(x, t.unsqueeze(0).to(device), ctx)
        x = scheduler.step(v_pred, t, x)

    # Decode latent to pixel space
    video = vae.decode(x)  # [1, 3, 16, 64, 64]

print(f"Generated video shape: {video.shape}")
print(f"Value range: [{video.min().item():.3f}, {video.max().item():.3f}]")

In [ ]:
# Display 8 frames from the generated video
video_np = video[0].detach().cpu()  # [3, 16, 64, 64]
num_display_frames = 8
frame_indices = torch.linspace(0, video_np.shape[1] - 1, num_display_frames).long()

fig, axes = plt.subplots(1, num_display_frames, figsize=(16, 2.5))
for i, idx in enumerate(frame_indices):
    frame = video_np[:, idx]  # [3, 64, 64]
    # Normalize from [-1, 1] to [0, 1] for display
    frame = (frame + 1) / 2
    frame = frame.clamp(0, 1)
    frame = frame.permute(1, 2, 0).numpy()  # [64, 64, 3]
    axes[i].imshow(frame)
    axes[i].set_title(f"Frame {idx.item()}")
    axes[i].axis('off')

fig.suptitle("Generated Video Frames (Quick Test)", fontsize=14)
plt.tight_layout()
plt.show()

print("\nTraining and inference test complete!")
print("See notebook 07_inference.ipynb for detailed generation and analysis.")